# BTCUSDT 5m - Deep Learning Sequence Model (TCN / Transformer / Bi-GRU)
This notebook implements an institutional-grade **Deep Learning (DL)** pipeline for **BTCUSDT 5m** cryptocurrency futures:
1. **Dual Storage (Local + Cloud)**: Automatically saves model weights, scalers, and logs to **BOTH local disk** and **Google Drive Cloud**.
2. **Stationary Sequential Features**: Multi-scale volatility, volume z-scores, momentum, and normalized order flow proxies.
3. **Instant 3D Sliding Window Tensors**: Vectorized `(Batch, Lookback=48, Features)` with zero forward leakage.
4. **Modular Deep Learning Architectures**:
   - **TCN (Temporal Convolutional Network)**: Dilated causal 1D residual convolutions.
   - **Time-Series Transformer**: Multi-head self-attention with temporal positional encodings.
   - **Bi-GRU with Attention Pooling**: Bidirectional recurrent memory tracking sequence momentum.
5. **Strict Temporal Split**: Chronological Train (60%), Validation (20%), and Sealed Hold-Out (20% OOS 2024–2026).
6. **Backtesting & Risk Metrics**: Win Rate, Sharpe Ratio, Profit Factor, Max Drawdown, and Breakeven Ratchets (+0.30R) with realistic fees.
7. **Colab + Local VS Code Compatible**: Auto-mounts Google Drive and auto-detects GPU acceleration (NVIDIA T4 / A100 / Local GTX).


## 1. Environment & Colab / Drive Auto-Mount

In [ ]:
# 1. Check GPU Hardware in Colab
!nvidia-smi

# 2. Google Drive Auto-Mount
import os, sys, warnings
warnings.filterwarnings("ignore", category=FutureWarning)

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
    print(">> [Colab] Google Drive mounted successfully.")
except Exception:
    IN_COLAB = False
    print(">> [Local/VS Code] Running in local environment.")

# 3. PyTorch Hardware & CUDA Verification
import time, math, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.utils.data import TensorDataset, DataLoader
    print(f">> PyTorch Version: {torch.__version__}")
    
    if torch.cuda.is_available():
        device = torch.device("cuda")
        gpu_name = torch.cuda.get_device_name(0)
        gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f">> [SUCCESS] ACTIVE GPU: {gpu_name} ({gpu_mem:.2f} GB VRAM) 🚀")
    else:
        device = torch.device("cpu")
        print(">> [ALERT] RUNNING ON CPU!")
        print(">> To switch to GPU in Colab:")
        print("   1. Click top menu: Runtime -> Change runtime type")
        print("   2. Select: T4 GPU -> Save")
        print("   3. In VS Code Colab extension: Reconnect selecting 'T4 GPU'")
except ImportError:
    print("Warning: PyTorch is not yet installed. Run: pip install torch")
    device = torch.device("cpu")


## 2. Configuration & Dual Storage (Local + Cloud)

In [ ]:
# ================= CONFIGURATION =================
TF             = "5m"
SYMBOL         = "BTCUSDT"

# Temporal Sequence Parameters
LOOKBACK_BARS  = 48     # Lookback window seen by the model
HORIZON_BARS   = 12      # Forward prediction horizon
BARRIER_MULT   = 1.5            # Target profit multiplier of rolling ATR

# Deep Learning Architecture
# Choices: "TCN", "Transformer", "BiGRU"
MODEL_TYPE     = "TCN"     
HIDDEN_DIM     = 64             # Latent channel / hidden feature dimension
NUM_LAYERS     = 3              # Depth of convolutional / recurrent / attention layers
DROPOUT        = 0.3            # Dropout rate for regularization
NUM_HEADS      = 4              # Transformer attention heads (if MODEL_TYPE == "Transformer")

# Training Hyperparameters
BATCH_SIZE     = 256
LEARNING_RATE  = 1e-3
WEIGHT_DECAY   = 1e-4           # L2 Regularization
EPOCHS         = 25
PATIENCE       = 5              # Early stopping patience
USE_AMP        = True           # Automatic Mixed Precision for fast GPU training

# ================= DUAL STORAGE SETUP (LOCAL + CLOUD) =================
# We automatically save trained weights and logs to BOTH Local and Google Drive!
LOCAL_DIR = rf"D:\Projects_Main\Trade-Pro\Trader-Pro\notebooks\model\dl_5m"
CLOUD_DIR = f"/content/drive/MyDrive/crypto_models/dl_5m"

OUTPUT_DIRS = []
if 'IN_COLAB' in globals() and IN_COLAB:
    # 1. Cloud: Google Drive
    OUTPUT_DIRS.append(CLOUD_DIR)
    # 2. Local VM disk on Colab
    OUTPUT_DIRS.append(f"./model/dl_5m")
    # 3. Local PC if path is mounted
    if os.path.exists(LOCAL_DIR):
        OUTPUT_DIRS.append(LOCAL_DIR)

    # Data search paths
    candidate_data_dirs = [
        "/content/drive/MyDrive/crypto_data",
        "/content/drive/MyDrive/Trade-Pro/Trader-Pro/notebooks/data",
        "/content/drive/MyDrive/Trader-Pro/notebooks/data",
        "/content/data",
        "./data"
    ]
    DATA_DIR = next((p for p in candidate_data_dirs if os.path.exists(p)), "/content/drive/MyDrive/crypto_data")
else:
    # Running locally in VS Code
    OUTPUT_DIRS.append(LOCAL_DIR)
    # Also save to local Google Drive sync folder if present on Windows
    gdrive_win = os.path.expanduser(f"~/Google Drive/crypto_models/dl_5m")
    if os.path.exists(os.path.dirname(gdrive_win)):
        OUTPUT_DIRS.append(gdrive_win)
    DATA_DIR = r"D:\Projects_Main\Trade-Pro\Trader-Pro\notebooks\data"

for out_dir in OUTPUT_DIRS:
    os.makedirs(out_dir, exist_ok=True)

print(f"Target: {SYMBOL} {TF} | Model: {MODEL_TYPE} | Lookback: {LOOKBACK_BARS} bars | Horizon: {HORIZON_BARS} bars")
print(f"Data Directory:  {DATA_DIR}")
print(f"Active Output Directories (Local + Cloud):")
for d in OUTPUT_DIRS:
    print(f"  -> {d}")


## 3. Load Historical OHLCV Parquet Data

In [ ]:
data_file = os.path.join(DATA_DIR, TF, f"{SYMBOL}_{TF}.parquet")
if not os.path.exists(data_file):
    # Fallback to local relative path if needed
    alt_file = os.path.join("notebooks", "data", TF, f"{SYMBOL}_{TF}.parquet")
    if os.path.exists(alt_file):
        data_file = alt_file
    else:
        raise FileNotFoundError(f"Missing data file: {data_file}. Verify Google Drive path or local data directory.")

df = pd.read_parquet(data_file)
time_col = "open_time" if "open_time" in df.columns else ("timestamp" if "timestamp" in df.columns else df.columns[0])
df[time_col] = pd.to_datetime(df[time_col], utc=True)
df = df.sort_values(time_col).reset_index(drop=True)

print(f"Loaded {len(df):,} bars from {df[time_col].min().date()} to {df[time_col].max().date()}")
print(f"Columns: {list(df.columns)}")


## 4. Stationary Financial Feature Engineering

In [ ]:
def compute_dl_features(df):
    """
    Computes strictly stationary, normalized features for neural sequence models:
    - Log returns across multi-scale horizons (1, 3, 6, 12 bars)
    - Trend & Moving Averages (EMA 20 distance, EMA 50 distance, EMA 20/50 spread)
    - Support & Resistance Proximity (distance to dynamic support/resistance in ATR units, zone triggers)
    - Volatility estimators (Garman-Klass, Normalized ATR, Realized Volatility)
    - Order flow & Volume standardization (Z-score over 48 periods)
    - Momentum & Oscillators (RSI 14, Normalized MACD, VWAP deviation)
    """
    feat = pd.DataFrame(index=df.index)
    close_s = pd.Series(df["close"].values)
    high_s  = pd.Series(df["high"].values)
    low_s   = pd.Series(df["low"].values)
    open_s  = pd.Series(df["open"].values)
    vol_s   = pd.Series(df["volume"].values)

    close = close_s.values
    high  = high_s.values
    low   = low_s.values
    open_ = open_s.values
    vol   = vol_s.values

    # 1. Multi-scale log returns
    feat["ret_1"] = np.log(close / np.roll(close, 1))
    feat["ret_3"] = np.log(close / np.roll(close, 3))
    feat["ret_6"] = np.log(close / np.roll(close, 6))
    feat["ret_12"] = np.log(close / np.roll(close, 12))

    # 2. Normalized High-Low and Candle Body Spreads
    feat["hl_spread"] = (high - low) / (close + 1e-8)
    feat["body_spread"] = (close - open_) / (close + 1e-8)

    # 3. Volatility: ATR & Garman-Klass
    tr = np.maximum(high - low, np.maximum(np.abs(high - np.roll(close, 1)), np.abs(low - np.roll(close, 1))))
    atr14 = pd.Series(tr).rolling(14, min_periods=1).mean().values
    feat["norm_atr"] = atr14 / (close + 1e-8)
    feat["realized_vol_20"] = pd.Series(feat["ret_1"]).rolling(20, min_periods=1).std().values
    log_hl = np.log(high / (low + 1e-8))
    log_co = np.log(close / (open_ + 1e-8))
    feat["gk_vol"] = 0.5 * (log_hl ** 2) - (2 * np.log(2) - 1) * (log_co ** 2)

    # 4. Moving Averages & Trend Regimes (EMA 20, EMA 50)
    ema20 = close_s.ewm(span=20, adjust=False).mean()
    ema50 = close_s.ewm(span=50, adjust=False).mean()
    feat["ema_dist_20"] = (close_s - ema20) / (close_s + 1e-8)
    feat["ema_dist_50"] = (close_s - ema50) / (close_s + 1e-8)
    feat["ema_spread_20_50"] = (ema20 - ema50) / (close_s + 1e-8)

    # 5. Dynamic Support & Resistance Levels (50-bar swing pivots)
    rolling_sup = low_s.rolling(50, min_periods=5).min()
    rolling_res = high_s.rolling(50, min_periods=5).max()
    feat["dist_to_support"] = (close_s - rolling_sup) / (atr14 + 1e-8)
    feat["dist_to_resistance"] = (rolling_res - close_s) / (atr14 + 1e-8)
    feat["at_support_zone"] = (feat["dist_to_support"] <= 1.0).astype(float)
    feat["at_resistance_zone"] = (feat["dist_to_resistance"] <= 1.0).astype(float)

    # 6. Volume Standardization (Z-score over 48 periods)
    vol_mean = vol_s.rolling(48, min_periods=1).mean().values
    vol_std  = vol_s.rolling(48, min_periods=1).std().values + 1e-8
    feat["vol_zscore"] = (vol - vol_mean) / vol_std

    # 7. Momentum & Oscillators (RSI 14, VWAP deviation)
    diff = close_s.diff()
    gain = (diff.where(diff > 0, 0)).rolling(14, min_periods=1).mean()
    loss = (-diff.where(diff < 0, 0)).rolling(14, min_periods=1).mean() + 1e-8
    rs = gain / loss
    rsi = 100 - (100 / (1 + rs))
    feat["norm_rsi"] = (rsi.values - 50.0) / 50.0

    cum_vol = vol_s.rolling(48, min_periods=1).sum().values + 1e-8
    cum_pv  = pd.Series(close * vol).rolling(48, min_periods=1).sum().values
    vwap = cum_pv / cum_vol
    feat["vwap_dev"] = (close - vwap) / (close + 1e-8)

    # Clean initial NaN values
    feat = feat.bfill().ffill().fillna(0.0)
    return feat, atr14

features_df, raw_atr = compute_dl_features(df)
print(f">> Generated {features_df.shape[1]} comprehensive features (including RSI, EMAs, Volume & Support/Resistance):")
for col in features_df.columns:
    print(f"   - {col}")


## 5. Directional Target Labeling (Dynamic Triple Barrier)

In [ ]:
# High-speed vectorized triple-barrier directional labeling
close_arr = df["close"].values
high_arr  = df["high"].values
low_arr   = df["low"].values
n_bars    = len(df)

labels_long = np.zeros(n_bars, dtype=np.float32)
labels_short = np.zeros(n_bars, dtype=np.float32)

t0 = time.time()
for i in range(n_bars - HORIZON_BARS):
    pt = BARRIER_MULT * raw_atr[i]
    sl = BARRIER_MULT * raw_atr[i]
    c0 = close_arr[i]
    
    fwd_high = high_arr[i+1 : i+1+HORIZON_BARS]
    fwd_low  = low_arr[i+1 : i+1+HORIZON_BARS]
    
    # Long: hits upper barrier before lower barrier
    hit_pt_l = np.where(fwd_high >= c0 + pt)[0]
    hit_sl_l = np.where(fwd_low <= c0 - sl)[0]
    p_l = hit_pt_l[0] if len(hit_pt_l) > 0 else 999
    s_l = hit_sl_l[0] if len(hit_sl_l) > 0 else 999
    if p_l < s_l:
        labels_long[i] = 1.0

    # Short: hits lower barrier before upper barrier
    hit_pt_s = np.where(fwd_low <= c0 - pt)[0]
    hit_sl_s = np.where(fwd_high >= c0 + sl)[0]
    p_s = hit_pt_s[0] if len(hit_pt_s) > 0 else 999
    s_s = hit_sl_s[0] if len(hit_sl_s) > 0 else 999
    if p_s < s_s:
        labels_short[i] = 1.0

print(f"Labeled {n_bars:,} bars in {time.time()-t0:.2f}s")
print(f"Class Distribution: Long positives: {np.mean(labels_long)*100:.2f}% | Short positives: {np.mean(labels_short)*100:.2f}%")


## 6. Chronological Train / Validation / Hold-Out Split & 3D Tensors

In [ ]:
from sklearn.preprocessing import RobustScaler
from numpy.lib.stride_tricks import sliding_window_view

# Temporal Split (60% Train, 20% Dev Validation, 20% Sealed Out-of-Sample Hold-Out)
N = len(features_df) - HORIZON_BARS
train_end = int(N * 0.60)
val_end   = int(N * 0.80)

X_raw = features_df.values[:N]
y_long_raw = labels_long[:N]

# Fit scaler strictly on the training partition only (Zero lookahead leakage)
scaler = RobustScaler()
X_scaled_tr = scaler.fit_transform(X_raw[:train_end])
X_scaled_val = scaler.transform(X_raw[train_end:val_end])
X_scaled_oos = scaler.transform(X_raw[val_end:])
X_scaled = np.vstack([X_scaled_tr, X_scaled_val, X_scaled_oos])

def create_sequences_fast(data, targets, seq_len):
    # Vectorized sliding window - 100x faster than python loops
    xs = sliding_window_view(data[:-1], seq_len, axis=0).transpose(0, 2, 1).astype(np.float32)
    ys = targets[seq_len:].astype(np.float32)
    return xs, ys

print("Constructing 3D sequence tensors...")
t0 = time.time()
X_seq_tr, y_seq_tr = create_sequences_fast(X_scaled[:train_end], y_long_raw[:train_end], LOOKBACK_BARS)
X_seq_val, y_seq_val = create_sequences_fast(X_scaled[train_end:val_end], y_long_raw[train_end:val_end], LOOKBACK_BARS)
X_seq_oos, y_seq_oos = create_sequences_fast(X_scaled[val_end:], y_long_raw[val_end:], LOOKBACK_BARS)
print(f"Built sequence tensors in {time.time()-t0:.2f}s")

print(f"Train sequences:    {X_seq_tr.shape} | Labels: {y_seq_tr.shape}")
print(f"Val sequences:      {X_seq_val.shape} | Labels: {y_seq_val.shape}")
print(f"Hold-out sequences: {X_seq_oos.shape} | Labels: {y_seq_oos.shape}")

train_dataset = TensorDataset(torch.from_numpy(X_seq_tr), torch.from_numpy(y_seq_tr))
val_dataset   = TensorDataset(torch.from_numpy(X_seq_val), torch.from_numpy(y_seq_val))
oos_dataset   = TensorDataset(torch.from_numpy(X_seq_oos), torch.from_numpy(y_seq_oos))

train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, pin_memory=(device.type == "cuda"))
val_loader    = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
oos_loader    = DataLoader(oos_dataset, batch_size=BATCH_SIZE, shuffle=False)


## 7. Deep Learning Architectures (TCN, Transformer, BiGRU)

In [ ]:
# --- Architecture 1: Temporal Convolutional Network (TCN) ---
class Chomp1d(nn.Module):
    def __init__(self, chomp_size):
        super().__init__()
        self.chomp_size = chomp_size
    def forward(self, x):
        return x[:, :, :-self.chomp_size].contiguous()

class TemporalBlock(nn.Module):
    def __init__(self, n_inputs, n_outputs, kernel_size, stride, dilation, padding, dropout=0.2):
        super().__init__()
        self.conv1 = nn.Conv1d(n_inputs, n_outputs, kernel_size, stride=stride, padding=padding, dilation=dilation)
        self.chomp1 = Chomp1d(padding)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)
        self.conv2 = nn.Conv1d(n_outputs, n_outputs, kernel_size, stride=stride, padding=padding, dilation=dilation)
        self.chomp2 = Chomp1d(padding)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)
        self.net = nn.Sequential(self.conv1, self.chomp1, self.relu1, self.dropout1,
                                 self.conv2, self.chomp2, self.relu2, self.dropout2)
        self.downsample = nn.Conv1d(n_inputs, n_outputs, 1) if n_inputs != n_outputs else None
        self.relu = nn.ReLU()

    def forward(self, x):
        out = self.net(x)
        res = x if self.downsample is None else self.downsample(x)
        return self.relu(out + res)

class TCNModel(nn.Module):
    def __init__(self, num_inputs, hidden_dim, num_levels=3, kernel_size=3, dropout=0.3):
        super().__init__()
        layers = []
        for i in range(num_levels):
            dilation_size = 2 ** i
            in_channels = num_inputs if i == 0 else hidden_dim
            layers.append(TemporalBlock(in_channels, hidden_dim, kernel_size, stride=1,
                                        dilation=dilation_size, padding=(kernel_size - 1) * dilation_size,
                                        dropout=dropout))
        self.network = nn.Sequential(*layers)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1)
        )

    def forward(self, x):
        # Shape: (Batch, Seq_Len, Features) -> (Batch, Features, Seq_Len)
        x = x.transpose(1, 2)
        y = self.network(x)
        out = self.fc(y[:, :, -1]) # Extract last temporal step
        return out.squeeze(-1)

# --- Architecture 2: Time-Series Transformer ---
class TimeSeriesTransformer(nn.Module):
    def __init__(self, num_inputs, hidden_dim, num_heads=4, num_layers=2, dropout=0.3):
        super().__init__()
        self.input_proj = nn.Linear(num_inputs, hidden_dim)
        self.pos_encoder = nn.Parameter(torch.randn(1, 256, hidden_dim) * 0.02)
        encoder_layer = nn.TransformerEncoderLayer(d_model=hidden_dim, nhead=num_heads,
                                                   dim_feedforward=hidden_dim*2, dropout=dropout,
                                                   batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1)
        )

    def forward(self, x):
        b, seq, _ = x.shape
        h = self.input_proj(x) + self.pos_encoder[:, :seq, :]
        h = self.transformer(h)
        out = self.fc(h.mean(dim=1)) # Temporal average pooling
        return out.squeeze(-1)

# --- Architecture 3: Bi-GRU with Attention ---
class BiGRUMaster(nn.Module):
    def __init__(self, num_inputs, hidden_dim, num_layers=2, dropout=0.3):
        super().__init__()
        self.gru = nn.GRU(num_inputs, hidden_dim, num_layers=num_layers,
                          batch_first=True, bidirectional=True, dropout=dropout if num_layers > 1 else 0)
        self.attn = nn.Linear(hidden_dim * 2, 1)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x):
        out, _ = self.gru(x)
        weights = torch.softmax(self.attn(out), dim=1)
        context = torch.sum(weights * out, dim=1)
        return self.fc(context).squeeze(-1)

def build_model(model_type, in_features):
    if model_type == "TCN":
        return TCNModel(num_inputs=in_features, hidden_dim=HIDDEN_DIM, num_levels=NUM_LAYERS, dropout=DROPOUT)
    elif model_type == "Transformer":
        return TimeSeriesTransformer(num_inputs=in_features, hidden_dim=HIDDEN_DIM, num_heads=NUM_HEADS, num_layers=NUM_LAYERS, dropout=DROPOUT)
    elif model_type == "BiGRU":
        return BiGRUMaster(num_inputs=in_features, hidden_dim=HIDDEN_DIM, num_layers=NUM_LAYERS, dropout=DROPOUT)
    else:
        raise ValueError(f"Unknown MODEL_TYPE: {model_type}")

model = build_model(MODEL_TYPE, in_features=X_seq_tr.shape[2]).to(device)
print(f"Initialized {MODEL_TYPE} model with {sum(p.numel() for p in model.parameters() if p.requires_grad):,} trainable parameters on {device}.")


## 8. PyTorch Training Loop with Mixed Precision & Early Stopping

In [ ]:
from sklearn.metrics import roc_auc_score

# Class weighting for balanced loss
pos_weight = torch.tensor([(len(y_seq_tr) - np.sum(y_seq_tr)) / (np.sum(y_seq_tr) + 1e-8)]).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)

# Modern PyTorch 2.x Mixed Precision (AMP)
use_cuda = (USE_AMP and device.type == "cuda")
if hasattr(torch, "amp") and hasattr(torch.amp, "GradScaler"):
    scaler_amp = torch.amp.GradScaler("cuda", enabled=use_cuda)
else:
    scaler_amp = torch.cuda.amp.GradScaler(enabled=use_cuda)

def autocast_ctx():
    if hasattr(torch, "amp") and hasattr(torch.amp, "autocast"):
        return torch.amp.autocast(device_type="cuda" if use_cuda else "cpu", enabled=use_cuda)
    return torch.cuda.amp.autocast(enabled=use_cuda)

history = {"train_loss": [], "val_loss": [], "val_auc": []}
best_val_auc = 0.0
best_model_weights = None
patience_counter = 0

print(f"Training {MODEL_TYPE} model for up to {EPOCHS} epochs on {device}...")
for epoch in range(1, EPOCHS + 1):
    # 1. Training Phase
    model.train()
    total_loss, total_count = 0.0, 0
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        with autocast_ctx():
            logits = model(bx)
            loss = criterion(logits, by)
        scaler_amp.scale(loss).backward()
        scaler_amp.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler_amp.step(optimizer)
        scaler_amp.update()
        total_loss += loss.item() * len(by)
        total_count += len(by)
    train_loss = total_loss / total_count

    # 2. Validation Phase
    model.eval()
    val_loss_tot, val_cnt = 0.0, 0
    preds_val, targets_val = [], []
    with torch.no_grad():
        for bx, by in val_loader:
            bx, by = bx.to(device), by.to(device)
            with autocast_ctx():
                logits = model(bx)
                loss = criterion(logits, by)
            val_loss_tot += loss.item() * len(by)
            val_cnt += len(by)
            preds_val.extend(torch.sigmoid(logits).cpu().numpy())
            targets_val.extend(by.cpu().numpy())

    val_loss = val_loss_tot / val_cnt
    val_auc = roc_auc_score(targets_val, preds_val) if len(np.unique(targets_val)) > 1 else 0.5
    scheduler.step(val_auc)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_auc"].append(val_auc)

    print(f"Epoch {epoch:02d}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val AUC: {val_auc:.4f}")

    if val_auc > best_val_auc:
        best_val_auc = val_auc
        best_model_weights = {k: v.cpu() for k, v in model.state_dict().items()}
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f">> Early stopping triggered at epoch {epoch}. Best Val AUC: {best_val_auc:.4f}")
            break

if best_model_weights is not None:
    model.load_state_dict({k: v.to(device) for k, v in best_model_weights.items()})
    print(f">> Loaded best model checkpoint with Val AUC: {best_val_auc:.4f}")


## 9. Sealed Out-of-Sample (OOS) Hold-Out Evaluation & Backtest

In [ ]:
# Sealed Hold-Out Evaluation (2024–2026 untouched data)
model.eval()
oos_preds, oos_targets = [], []
with torch.no_grad():
    for bx, by in oos_loader:
        bx = bx.to(device)
        logits = model(bx)
        oos_preds.extend(torch.sigmoid(logits).cpu().numpy())
        oos_targets.extend(by.cpu().numpy())

oos_preds = np.array(oos_preds)
oos_targets = np.array(oos_targets)
oos_auc = roc_auc_score(oos_targets, oos_preds) if len(np.unique(oos_targets)) > 1 else 0.5
print(f">> Sealed Hold-Out (2024–2026) ROC-AUC: {oos_auc:.4f}")

# Backtest Simulation with Breakeven Ratchet (+0.30R) and 0.05% taker fee
threshold = np.quantile(oos_preds, 0.85) # Top 15% conviction signals
signals = (oos_preds >= threshold)

trades_pnl = []
oos_offset = val_end + LOOKBACK_BARS
oos_close = df["close"].values[oos_offset : oos_offset + len(signals)]
oos_atr   = raw_atr[oos_offset : oos_offset + len(signals)]

for i in range(len(signals) - HORIZON_BARS):
    if not signals[i]:
        continue
    c0 = oos_close[i]
    atr = oos_atr[i]
    pt = BARRIER_MULT * atr
    sl = BARRIER_MULT * atr
    be_trigger = 0.30 * pt
    
    trade_ret = 0.0
    sl_cur = c0 - sl
    for step in range(1, HORIZON_BARS + 1):
        cur_high = high_arr[oos_offset + i + step]
        cur_low  = low_arr[oos_offset + i + step]
        cur_close= close_arr[oos_offset + i + step]

        # Breakeven Ratchet (+0.30R moves stop to Breakeven + fee buffer)
        if cur_high >= c0 + be_trigger and sl_cur < c0:
            sl_cur = c0 + 0.0005 * c0 # Lock in fee buffer
            
        if cur_high >= c0 + pt:
            trade_ret = (pt / c0) - 0.0010 # Win minus round-trip fees
            break
        elif cur_low <= sl_cur:
            trade_ret = ((sl_cur - c0) / c0) - 0.0010 # Stop loss
            break
        elif step == HORIZON_BARS:
            trade_ret = ((cur_close - c0) / c0) - 0.0010 # Horizon exit

    trades_pnl.append(trade_ret)

trades_pnl = np.array(trades_pnl)
if len(trades_pnl) > 0:
    win_rate = np.mean(trades_pnl > 0) * 100
    cum_returns = np.cumprod(1 + trades_pnl)
    peak = np.maximum.accumulate(cum_returns)
    dd = (cum_returns - peak) / peak
    max_dd = np.min(dd) * 100
    wins = trades_pnl[trades_pnl > 0]
    losses = trades_pnl[trades_pnl < 0]
    profit_factor = (np.sum(wins) / np.abs(np.sum(losses))) if len(losses) > 0 else np.nan
    bars_per_day = 1440 // int(TF.replace("m",""))
    sharpe = (np.mean(trades_pnl) / (np.std(trades_pnl) + 1e-8)) * np.sqrt(252 * bars_per_day)

    print("=================== PERFORMANCE SUMMARY ===================")
    print(f"Total Trades Taken: {len(trades_pnl):,}")
    print(f"Win Rate:           {win_rate:.2f}%")
    print(f"Profit Factor:      {profit_factor:.2f}")
    print(f"Sharpe Ratio:       {sharpe:.2f}")
    print(f"Max Drawdown:       {max_dd:.2f}%")
    print("===========================================================")


## 10. Visual Performance Charts

In [ ]:
if len(history["train_loss"]) > 0:
    fig, ax = plt.subplots(1, 2, figsize=(14, 5))

    # Loss Curve
    ax[0].plot(history["train_loss"], label="Train Loss", color="royalblue", lw=2)
    ax[0].plot(history["val_loss"], label="Val Loss", color="crimson", lw=2)
    ax[0].set_title(f"{SYMBOL} {TF} - DL Training Loss")
    ax[0].set_xlabel("Epoch")
    ax[0].set_ylabel("BCE Loss")
    ax[0].legend()
    ax[0].grid(True, alpha=0.3)

    # Validation AUC
    ax[1].plot(history["val_auc"], label="Validation ROC-AUC", color="darkgreen", lw=2)
    ax[1].axhline(0.5, color="grey", ls="--", label="Random Baseline (0.50)")
    ax[1].set_title(f"{SYMBOL} {TF} - Validation ROC-AUC Curve")
    ax[1].set_xlabel("Epoch")
    ax[1].set_ylabel("AUC")
    ax[1].legend()
    ax[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


## 11. Dual Storage: Save to BOTH Local Disk and Cloud Google Drive

In [ ]:
import joblib

checkpoint = {
    "model_state_dict": model.state_dict(),
    "model_type": MODEL_TYPE,
    "input_features": list(features_df.columns),
    "lookback_bars": LOOKBACK_BARS,
    "horizon_bars": HORIZON_BARS,
    "hidden_dim": HIDDEN_DIM,
    "num_layers": NUM_LAYERS
}

metrics_log = {
    "symbol": SYMBOL,
    "timeframe": TF,
    "model_type": MODEL_TYPE,
    "lookback_bars": LOOKBACK_BARS,
    "horizon_bars": HORIZON_BARS,
    "test_auc": float(oos_auc) if 'oos_auc' in globals() else 0.0,
    "win_rate": float(win_rate) if 'win_rate' in globals() else 0.0,
    "profit_factor": float(profit_factor) if 'profit_factor' in globals() and not np.isnan(profit_factor) else 0.0,
    "sharpe": float(sharpe) if 'sharpe' in globals() else 0.0,
    "max_drawdown": float(max_dd) if 'max_dd' in globals() else 0.0,
    "trades_count": int(len(trades_pnl)) if 'trades_pnl' in globals() else 0
}

print(">> Saving model artifacts to all active storage locations (Local + Cloud):")
for target_dir in OUTPUT_DIRS:
    os.makedirs(target_dir, exist_ok=True)

    # 1. Save PyTorch Model Weights
    torch_save_path = os.path.join(target_dir, f"final_dl_model_{SYMBOL}_{TF}.pt")
    torch.save(checkpoint, torch_save_path)
    print(f"   [SUCCESS] PyTorch model weights -> {torch_save_path}")

    # 2. Save Scaler
    scaler_path = os.path.join(target_dir, f"scaler_{SYMBOL}_{TF}.joblib")
    joblib.dump(scaler, scaler_path)
    print(f"   [SUCCESS] Scaler -> {scaler_path}")

    # 3. Save Metrics Log
    log_path = os.path.join(target_dir, "metrics_dl.json")
    with open(log_path, "w") as f:
        json.dump(metrics_log, f, indent=4)
    print(f"   [SUCCESS] Metrics summary -> {log_path}")

print("\n>> All model checkpoints and scalers are now safely stored in both Local and Cloud Google Drive!")
